In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from Bio import SeqIO
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/Screening_and_application


In [ ]:
df_test = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_test_with_ESM1b_ts.pkl")
)
df_test = df_test.loc[df_test["ESM1b"] != ""]
df_test.reset_index(inplace=True, drop=True)

df_train = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_train_with_ESM1b_ts.pkl")
)
df_train = df_train.loc[df_train["ESM1b"] != ""]
df_train.reset_index(inplace=True, drop=True)

/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)
/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


In [ ]:
p450plant0 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "p450plant0.pkl")
)
p450plant1 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "p450plant1.pkl")
)
p450plant2 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "p450plant2.pkl")
)
p450plant3 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "p450plant3.pkl")
)
p450plant4 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "p450plant4.pkl")
)


p450plant = pd.concat(
    [p450plant0, p450plant1, p450plant2, p450plant3, p450plant4], ignore_index=True
)

In [4]:
p450plant

,enzyme,substrate,Binding,ECFP,ESM1b
0,CYP71AY5,GEI,1,0000000000000000000000000000000001000000000010...,"[-0.032707780599594116, 0.15956459939479828, -..."
1,CYP85A2,CAT,1,0100000010000000000000000000010001001000000000...,"[-0.12464731186628342, 0.25692933797836304, -0..."
2,CYP716A94,BAM,1,0000000000000000000000000000000101001000000000...,"[-0.1793026179075241, 0.2773324251174927, 0.02..."
3,CYP80B2,NME,1,0001000000000000000000000100000001000000100000...,"[-0.05789301171898842, 0.1279979795217514, -0...."
4,CYP85A69,DOC,1,0100000010000000000000000000000001001000000000...,"[-0.077804334461689, 0.2830597162246704, -0.00..."
...,...,...,...,...,...
1540,CYP82E1,CLA,0,0000101000000000000000000000000101001000000000...,"[0.014755867421627045, 0.14731034636497498, -0..."
1541,CYP706V2,Marmesin,0,0000000000000000000000000000000001000000000000...,"[-0.041090771555900574, 0.14285863935947418, 0..."
1542,CYP706V2,SCO,0,0001000000000000000000000000000001000000100000...,"[-0.041090771555900574, 0.14285863935947418, 0..."
1543,CYP71D495,BYB,0,0100000010000000000000000010010001001000000000...,"[-0.08674226701259613, 0.1240856796503067, -0...."


In [ ]:
deletedata = pd.read_csv(
    our_data + "screening1/P450_Substrate.txt",
    sep="\t",
    header=None,
    names=["enzyme", "substrate", "sequence"],
)

deletedata = deletedata.sample(frac=1, random_state=42)


! python distinctonly.py --input "/home/hanxd/Repositories/ESP/data/our_data/screening1/testdata.fasta" --output "/home/hanxd/Repositories/ESP/data/our_data/screening1/testdata_q.fasta"

In [ ]:
fasta_file = our_data + "screening1/testdata_q.fasta"
sequences = SeqIO.to_dict(SeqIO.parse(fasta_file, "fasta"))

In [ ]:
for index, row in deletedata.iterrows():
    nowenzyme = row["enzyme"]
    nowsubstrate = row["substrate"]
    try:
        target_sequence = sequences[nowenzyme].seq
    except:
        target_sequence = ""
    deletedata.loc[index, "sequence"] = str(target_sequence)

In [ ]:
deletedata = deletedata[deletedata["sequence"] != ""]

has_null_values = deletedata.isna().any().any()
has_empty_strings = deletedata.applymap(lambda x: x == "")
if has_empty_strings.any().any():
    print("DataFrame contains empty strings.")
if has_null_values:
    print("DataFrame contains empty values.")

In [ ]:
deletedata["ESM1b"] = ""
deletedata["ECFP"] = ""

rep_dict = join(CURRENT_DIR, "..", "data", "our_data", "screening1/esm/")

for ind in deletedata.index:
    esms = torch.load(rep_dict + str(deletedata["enzyme"][ind]) + ".pt")
    sdf_file_path = (
        our_data + "screening1/SDF_300/" + deletedata["substrate"][ind] + ".sdf"
    )
    mol = Chem.MolFromMolFile(sdf_file_path)
    ecfpso = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024).ToBitString()
    deletedata["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    deletedata["ECFP"][ind] = ecfpso

In [ ]:
deletedata["Binding"] = 1

has_null_values = deletedata.isna().any().any()
has_empty_strings = deletedata.applymap(lambda x: x == "")
if has_empty_strings.any().any():
    print("DataFrame contains empty strings.")
if has_null_values:
    print("DataFrame contains empty values.")

In [ ]:
deletedata.to_pickle(our_data + "screening1/" + "deletedata.pkl")

In [ ]:
filtered_rows = p450plant[
    (p450plant["Binding"] == 1) & p450plant["enzyme"].isin(deletedata["enzyme"])
]

p450plant_filtered = p450plant.drop(filtered_rows.index)

p450plant_filtered.reset_index(drop=True, inplace=True)

In [ ]:
p450plant_filtered[p450plant_filtered["enzyme"] == "CYP701A26"]

,enzyme,substrate,Binding,ECFP,ESM1b
239,CYP701A26,SAB,0,0100000000001000000000000000000001011000000010...,"[-0.025068089365959167, 0.14765217900276184, -..."
240,CYP701A26,CSL,0,0000000000000000000000000000000001000000000000...,"[-0.025068089365959167, 0.14765217900276184, -..."


## train

In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


train_X, train_y = create_input_and_output_data(df=df_train)
test_X, test_y = create_input_and_output_data(df=df_test)
train_new_X, train_new_y = create_input_and_output_data(df=p450plant_filtered)
# train_new_X, train_new_y =  create_input_and_output_data(df = p450plant)

train_X = np.concatenate([train_X, train_new_X])
train_y = np.concatenate([train_y, train_new_y])

feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

In [ ]:
param = {
    "learning_rate": 0.31553117247348733,
    "max_delta_step": 1.7726044219753656,
    "max_depth": 10,
    "min_child_weight": 1.3845040588450772,
    "num_rounds": 342.68325188584106,
    "reg_alpha": 0.531395259755843,
    "reg_lambda": 3.744980563764689,
    "weight": 0.26187490421514203,
}


num_round = param["num_rounds"]
param["objective"] = "binary:logistic"

weights = np.array(
    [param["weight"] if binding == 0 else 1.0 for binding in np.array(train_y)]
)

del param["num_rounds"]
del param["weight"]

dtrain = xgb.DMatrix(
    np.array(train_X),
    weight=weights,
    label=np.array(train_y),
    feature_names=feature_names,
)
dtest = xgb.DMatrix(
    np.array(test_X), label=np.array(test_y), feature_names=feature_names
)
dtrain_p450 = xgb.DMatrix(
    np.array(train_new_X), label=np.array(train_new_y), feature_names=feature_names
)
evallist = [(dtrain, "train"), (dtrain_p450, "dtrain_p450")]
bst = xgb.train(param, dtrain, int(num_round), evallist, verbose_eval=10)
y_test_pred = np.round(bst.predict(dtest))
acc_test = np.mean(y_test_pred == np.array(test_y))
roc_auc = roc_auc_score(np.array(test_y), bst.predict(dtest))

print("Accuracy on test set: %s, ROC-AUC score for test set: %s" % (acc_test, roc_auc))

[0]	train-error:0.26954	dtrain_p450-error:0.55937
[10]	train-error:0.13136	dtrain_p450-error:0.39578
[20]	train-error:0.08620	dtrain_p450-error:0.27045
[30]	train-error:0.06020	dtrain_p450-error:0.18865
[40]	train-error:0.04376	dtrain_p450-error:0.13325
[50]	train-error:0.03536	dtrain_p450-error:0.10488
[60]	train-error:0.02814	dtrain_p450-error:0.07454
[70]	train-error:0.02276	dtrain_p450-error:0.05475
[80]	train-error:0.01933	dtrain_p450-error:0.05211
[90]	train-error:0.01653	dtrain_p450-error:0.04222
[100]	train-error:0.01398	dtrain_p450-error:0.03364
[110]	train-error:0.01181	dtrain_p450-error:0.02770
[120]	train-error:0.01021	dtrain_p450-error:0.02638
[130]	train-error:0.00880	dtrain_p450-error:0.02177
[140]	train-error:0.00758	dtrain_p450-error:0.01979
[150]	train-error:0.00647	dtrain_p450-error:0.01781
[160]	train-error:0.00555	dtrain_p450-error:0.01715
[170]	train-error:0.00483	dtrain_p450-error:0.01451
[180]	train-error:0.00431	dtrain_p450-error:0.01385
[190]	train-error:0.003

In [ ]:
pickle.dump(bst, open(join(our_data + "screening1/deletedatamodel.dat"), "wb"))

In [ ]:
len(p450plant) - len(train_new_y)

29